# 05.04 · Images & PDFs as input

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Send images to vision models (base64 data URLs / file URLs)
- [ ] Two ways to handle PDFs: extract text locally vs send the document to a model that reads PDFs natively
- [ ] Understand image token costs and resolution trade-offs

## Resources
- [OpenAI – Images and vision](https://platform.openai.com/docs/guides/images-vision) — free
- [Gemini – Image understanding](https://ai.google.dev/gemini-api/docs/image-understanding) · [Document understanding](https://ai.google.dev/gemini-api/docs/document-processing) — free
- [Anthropic – Vision](https://docs.anthropic.com/en/docs/build-with-claude/vision) — free

**Vision model needed for sections 1–2.** Options: free Gemini key (`GEMINI_API_KEY`), or local `ollama pull gemma3:4b`
(~3 GB) / `qwen2.5vl:3b`. The default `qwen3:1.7b` is text-only, so those cells will tell you what's missing.

In [ ]:
import base64
import io
import os

import matplotlib.pyplot as plt
from pydantic import BaseModel

from aieng import DATA
from aieng.llm import client

if os.getenv("GEMINI_API_KEY"):
    vc, VISION_MODEL = client("gemini")
else:
    vc, _ = client("ollama")
    VISION_MODEL = os.getenv("OLLAMA_VISION_MODEL", "gemma3:4b")
print("vision model:", VISION_MODEL)

## 1. Make a test image: a receipt
We draw one so the notebook is self-contained. Swap in a photo of a real receipt for the exercise.

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 4))
ax.axis("off")
lines = ["SUPERMERCADO SOL", "Av. Arequipa 123, Lima", "2026-06-02  18:41", "", "Leche 1L        x2   9.80",
         "Pan integral    x1   7.50", "Palta           x3  12.60", "Cafe molido     x1  24.90", "", "TOTAL      S/  54.80"]
for i, line in enumerate(lines):
    ax.text(0.02, 0.95 - i * 0.09, line, family="monospace", fontsize=9, transform=ax.transAxes)
buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=150, bbox_inches="tight"); plt.show()
png = buf.getvalue()
(DATA / "receipt.png").write_bytes(png)
print(f"{len(png) / 1024:.0f} KB")

## 2. Send it (OpenAI-compatible format)
A message's `content` becomes a **list of parts**: text parts and `image_url` parts. Local images go as base64 data URLs.

In [ ]:
def image_part(png_bytes, mime="image/png"):
    return {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{base64.b64encode(png_bytes).decode()}"}}


class Item(BaseModel):
    name: str
    quantity: int
    price: float


class Receipt(BaseModel):
    store: str
    date: str
    items: list[Item]
    total: float


try:
    r = vc.chat.completions.parse(model=VISION_MODEL, response_format=Receipt, temperature=0, messages=[{"role": "user", "content": [
        {"type": "text", "text": "Extract this receipt."}, image_part(png)]}])
    print(r.choices[0].message.parsed)
except Exception as e:
    print(f"Vision call failed ({type(e).__name__}: {str(e)[:150]}).\n"
          "Add GEMINI_API_KEY to .env or run `ollama pull gemma3:4b`, then rerun.")

**Image cost**: images are converted to tokens (roughly by tile/patch count; often ~250–1,500 tokens per image depending on provider and
resolution). Downscale images to the smallest size that keeps the detail you need. Many APIs have a `detail: "low"` option.

## 3. PDFs, route A: extract text locally
Cheap, fast, works with any text model. Loses layout, tables and images; fails on scanned PDFs (need OCR).

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader

pdf_path = DATA / "report.pdf"
with PdfPages(pdf_path) as pdf:
    for page, body in enumerate([
        "Q2 Report\n\nRevenue grew 18% quarter over quarter to $4.2M.\nChurn fell from 3.1% to 2.4%.",
        "Risks\n\n1. Dependence on a single cloud provider.\n2. Hiring for ML roles is slow.\n3. LLM API costs rose 40%.",
    ]):
        f = plt.figure(figsize=(8.5, 11)); f.text(0.1, 0.85, body, fontsize=14, va="top"); pdf.savefig(f); plt.close(f)

text = "\n\n".join(p.extract_text() for p in PdfReader(pdf_path).pages)
print(text)

c, MODEL = client()
extra = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
r = c.chat.completions.create(model=MODEL, **extra, messages=[{"role": "user", "content": f"Summarise in 2 bullet points:\n\n{text}"}])
print("\n", r.choices[0].message.content)

**Check the summary against the source.** Small models often subtly change facts — e.g. "quarter over quarter" becoming
"year over year" — or add claims that aren't there ("driven by improved performance"). Faithfulness evals (chapter 10) catch this.

## 4. PDFs, route B: native document understanding
Gemini, Claude and OpenAI accept PDFs directly and "see" pages as images + text: tables, charts and scans work.
More expensive (each page costs image tokens). Gemini native SDK example (runs if you have the key):

In [ ]:
if os.getenv("GEMINI_API_KEY"):
    from google import genai
    from google.genai import types

    g = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    r = g.models.generate_content(model="gemini-flash-latest", contents=[
        types.Part.from_bytes(data=pdf_path.read_bytes(), mime_type="application/pdf"), "List the risks mentioned."])
    print(r.text)
else:
    print("GEMINI_API_KEY not set — skipping native PDF demo")

Rule of thumb: text-heavy, digital PDFs → extract locally (route A, or better parsers in 08.02). Scans, tables, charts, forms →
native multimodal (route B) or a dedicated parser (Docling, LlamaParse).

## Exercises
1. Photograph a real receipt; extract it with the `Receipt` schema. Measure accuracy over 5 receipts.
2. Resize the image to 25%, 50%, 100% before sending. Where does extraction start failing? How do the input token counts change?
3. Find a PDF with a table. Compare route A (pypdf) vs route B (native) on answering a question about the table.

In [ ]:
# your code here